# Part1 Mlp Tabular

Notebook version of `part1_mlp_tabular.py`. Run the cells from top to bottom.

In [ ]:
from pathlib import Path

import numpy as np

import pandas as pd

import torch

from sklearn.datasets import fetch_openml

from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from torch import nn

from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = Path.cwd() / "data"

ADULT_DATA_FILE = DATA_DIR / "adult_income.csv"

ADULT_OPENML_ID = 1590

ADULT_SOURCE_URL = "https://archive.ics.uci.edu/dataset/2/adult"

TARGET_COLUMN = "income"


## Shared utilities

Embedded so this notebook runs without `utils.py`.


In [ ]:
"""Utilitaires communs aux trois parties du projet Deep Learning."""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"

for directory in (FIGURES_DIR, METRICS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def get_device():
    """Return the best device available to PyTorch."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def plot_history(history, title, output_path):
    plt.figure(figsize=(7, 4))
    for name, values in history.items():
        plt.plot(range(1, len(values) + 1), values, label=name.replace("_", " "))
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_confusion_matrix(matrix, class_names, title, output_path):
    plt.figure(figsize=(5, 4))
    plt.imshow(matrix, interpolation="nearest", cmap="Blues")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=30, ha="right")
    plt.yticks(ticks, class_names)
    threshold = matrix.max() / 2 if matrix.size else 0
    for row, column in np.ndindex(matrix.shape):
        plt.text(
            column,
            row,
            str(matrix[row, column]),
            ha="center",
            va="center",
            color="white" if matrix[row, column] > threshold else "black",
        )
    plt.ylabel("Classe reelle")
    plt.xlabel("Classe predite")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def _json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Type non serialisable: {type(value).__name__}")


def save_json(payload, output_path):
    with Path(output_path).open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2, default=_json_default)



In [ ]:
class CustomMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(0.15)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.dropout(x)
        x = self.relu2(self.fc2(x))
        return self.fc3(x)


In [ ]:
def build_sequential_mlp(input_dim, hidden_dim=64, output_dim=2):
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(hidden_dim, hidden_dim // 2),
        nn.ReLU(),
        nn.Linear(hidden_dim // 2, output_dim),
    )


In [ ]:
def initialize_model(model, strategy):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            if strategy == "gaussian":
                nn.init.normal_(module.weight, mean=0.0, std=0.05)
                nn.init.zeros_(module.bias)
            elif strategy == "constant":
                nn.init.constant_(module.weight, 0.01)
                nn.init.zeros_(module.bias)
            elif strategy == "xavier":
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
            else:
                raise ValueError(f"Strategie inconnue: {strategy}")


In [ ]:
def load_adult_income():
    """Load Adult Income from the local project cache or OpenML.

    The local CSV makes later executions reproducible.  On the first run, the
    dataset is downloaded from OpenML (dataset id 1590) and copied into data/.
    """
    if ADULT_DATA_FILE.exists():
        data = pd.read_csv(ADULT_DATA_FILE)
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        try:
            adult = fetch_openml(
                data_id=ADULT_OPENML_ID,
                as_frame=True,
                data_home=DATA_DIR / "openml_cache",
                parser="auto",
            )
        except Exception as error:
            raise RuntimeError(
                "Impossible de telecharger Adult Income. Verifiez votre connexion "
                f"ou placez un fichier CSV dans {ADULT_DATA_FILE}."
            ) from error
        data = adult.data.copy()
        data[TARGET_COLUMN] = adult.target.astype(str)
        data.to_csv(ADULT_DATA_FILE, index=False)

    if TARGET_COLUMN not in data.columns:
        # The OpenML version calls the target 'class'; accepting it makes a
        # manually downloaded file usable without changing the code.
        if "class" not in data.columns:
            raise ValueError(f"La colonne cible '{TARGET_COLUMN}' est absente du dataset.")
        data = data.rename(columns={"class": TARGET_COLUMN})
    return data


In [ ]:
def load_tabular_data(batch_size=64):
    data = load_adult_income()
    data = data.replace("?", np.nan)
    data = data.replace(r"^\s*\?\s*$", np.nan, regex=True)

    # The target can have a trailing dot in some UCI files (for example '>50K.').
    target = data[TARGET_COLUMN].astype(str).str.strip().str.rstrip(".")
    valid_targets = {"<=50K": 0, ">50K": 1}
    data = data.loc[target.isin(valid_targets)].copy()
    y = target.loc[data.index].map(valid_targets).astype(np.int64)
    x = data.drop(columns=[TARGET_COLUMN])

    x_train, x_temp, y_train, y_temp = train_test_split(
        x, y, test_size=0.30, random_state=42, stratify=y
    )
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    numeric_columns = x_train.select_dtypes(include=["number"]).columns.tolist()
    categorical_columns = [column for column in x_train.columns if column not in numeric_columns]
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_columns,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical_columns,
            ),
        ],
        verbose_feature_names_out=False,
    )

    x_train = preprocessor.fit_transform(x_train).astype(np.float32)
    x_val = preprocessor.transform(x_val).astype(np.float32)
    x_test = preprocessor.transform(x_test).astype(np.float32)

    train_loader = DataLoader(TensorDataset(torch.tensor(x_train), torch.tensor(y_train.to_numpy())), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(x_val), torch.tensor(y_val.to_numpy())), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(torch.tensor(x_test), torch.tensor(y_test.to_numpy())), batch_size=batch_size)
    return train_loader, val_loader, test_loader, x_train.shape[1], ["<=50K", ">50K"], preprocessor


In [ ]:
def train_classifier(model, train_loader, val_loader, device, epochs=25, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}
    best_state = None
    best_val_loss = float("inf")

    model.to(device)
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x_batch.size(0)
        history["train_loss"].append(total_loss / len(train_loader.dataset))

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                loss = criterion(model(x_batch), y_batch)
                val_loss += loss.item() * x_batch.size(0)
        val_loss /= len(val_loader.dataset)
        history["val_loss"].append(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return history


In [ ]:
def evaluate_classifier(model, data_loader, device):
    model.eval()
    predictions = []
    targets = []
    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            logits = model(x_batch.to(device))
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            targets.extend(y_batch.numpy().tolist())

    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, predictions, average="binary", pos_label=1, zero_division=0
    )
    return {
        "accuracy": accuracy_score(targets, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": confusion_matrix(targets, predictions),
    }


In [ ]:
def inspect_parameters(model):
    named = {
        name: {"shape": list(param.shape), "requires_grad": param.requires_grad}
        for name, param in model.named_parameters()
    }
    state = {name: list(value.shape) for name, value in model.state_dict().items()}
    return {"named_parameters": named, "state_dict_shapes": state}


In [ ]:
def run_part1():
    device = get_device()
    train_loader, val_loader, test_loader, input_dim, class_names, preprocessor = load_tabular_data()
    strategies = ["gaussian", "constant", "xavier"]
    rows = []
    best_model = None
    best_f1 = -1.0

    for model_type in ["sequential", "custom"]:
        for strategy in strategies:
            model = build_sequential_mlp(input_dim) if model_type == "sequential" else CustomMLP(input_dim)
            initialize_model(model, strategy)
            history = train_classifier(model, train_loader, val_loader, device)
            metrics = evaluate_classifier(model, test_loader, device)
            rows.append({
                "model": model_type,
                "initialization": strategy,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
            })
            if metrics["f1"] > best_f1:
                best_f1 = metrics["f1"]
                best_model = model
                torch.save(model.state_dict(), MODELS_DIR / "best_mlp_adult_income.pt")
                plot_history(history, "Partie I - apprentissage MLP (Adult Income)", FIGURES_DIR / "part1_mlp_history.png")
                plot_confusion_matrix(
                    metrics["confusion_matrix"],
                    class_names,
                    "Partie I - matrice de confusion MLP (Adult Income)",
                    FIGURES_DIR / "part1_mlp_confusion.png",
                )

    reloaded = CustomMLP(input_dim) if isinstance(best_model, CustomMLP) else build_sequential_mlp(input_dim)
    reloaded.load_state_dict(torch.load(MODELS_DIR / "best_mlp_adult_income.pt", map_location=device))
    reloaded.to(device)
    reloaded_metrics = evaluate_classifier(reloaded, test_loader, device)

    pd.DataFrame(rows).to_csv(METRICS_DIR / "part1_mlp_results.csv", index=False)
    save_json(inspect_parameters(best_model), METRICS_DIR / "part1_parameter_inspection.json")
    save_json({
        "dataset": "Adult Income / Census Income",
        "source_url": ADULT_SOURCE_URL,
        "openml_data_id": ADULT_OPENML_ID,
        "target": TARGET_COLUMN,
        "positive_class": ">50K",
        "device": str(device),
        "input_features_after_one_hot_encoding": input_dim,
        "best_f1": best_f1,
        "reloaded_model_f1": reloaded_metrics["f1"],
        "preprocessing": str(preprocessor),
    }, METRICS_DIR / "part1_summary.json")
    print(pd.DataFrame(rows))
    print(f"Meilleur F1 (>50K): {best_f1:.4f}; F1 apres rechargement: {reloaded_metrics['f1']:.4f}")


## Run the experiment


In [ ]:
if __name__ == "__main__":
    run_part1()
